# RAG 评估

本 notebook 对应课件里的评估体系部分，重点演示：
- 检索层：Recall / MRR / nDCG
- 生成层：RAGAS 三指标（Answer Relevancy / Faithfulness / Context Precision）


In [12]:
# 环境
import os, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

PROJECT_ROOT = Path('/Users/mengbai/Documents/AI-training/RAG_project')
CHROMA_DIR = PROJECT_ROOT / 'data/chroma'
COLLECTION = 'autel_annual_report_2024'

env_path = PROJECT_ROOT / '.env'
if not env_path.exists():
    env_path = PROJECT_ROOT.parent / '.env'
load_dotenv(env_path, override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
openai_base_url = os.getenv('OPENAI_BASE_URL')
emb = OpenAIEmbeddings(model=os.getenv('EMBED_MODEL'), api_key=openai_api_key, base_url=openai_base_url)
vs = Chroma(collection_name=COLLECTION, embedding_function=emb, persist_directory=str(CHROMA_DIR))
llm = ChatOpenAI(model=os.getenv('CHAT_MODEL'), temperature=0, api_key=openai_api_key, base_url=openai_base_url)
print('OK')


OK


In [13]:
# 检索函数
def retrieve(q, k=6):
    return vs.similarity_search(q, k=k)

EVAL_ITEMS = [
    {
        'question': '道通科技各产品线的毛利率分别是多少？',
        'reference': '2024年主营业务分产品毛利率包括：汽车综合诊断产品 54.42%，TPMS产品 55.33%，软件升级服务 99.46%，ADAS产品 60.07%，其他产品 34.54%，智能充电网络 37.27%。',
    },
    {
        'question': '公司前五大客户占比多少？',
        'reference': '2024年前五名客户销售额为98,127.13万元，占年度销售总额24.95%，其中前五名客户中关联方销售额为0万元。',
    },
    {
        'question': '研发费用和营收增长的关系是什么？',
        'reference': '2024年研发投入合计为680,027,155.85元，同比增长14.07%，占营业收入17.29%；同期营业收入为3,932,256,447.46元，同比增长20.95%。两者均增长，但营收增速高于研发投入增速。',
    },
]
TOPK = 6
print('OK')


OK


## Recall / MRR / nDCG


In [9]:
# Recall / MRR / nDCG
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

JUDGE_PROMPT = ChatPromptTemplate.from_messages([
    ('system', '判断是否相关。只输出 {{"relevant": 0|1}}。'),
    ('human', '问题：{q}\n\n证据：{c}')
])
parser = JsonOutputParser()

def is_relevant(q, c):
    try:
        r = (JUDGE_PROMPT | llm | parser).invoke({'q': q, 'c': c[:1200]})
        return int(r.get('relevant', 0))
    except:
        return 0

def recall_at_k(labels):
    return 1.0 if any(labels) else 0.0

def mrr_fn(labels):
    for i, v in enumerate(labels, 1):
        if v == 1: return 1.0 / i
    return 0.0

def ndcg_fn(labels):
    d = sum((2**v-1)/(i+1) for i,v in enumerate(labels))
    i = sum((2**v-1)/(i+1) for i,v in enumerate(sorted(labels,reverse=True)))
    return d/i if i>0 else 0.0

print('='*50)
recalls, mrrs, ndcgs = [], [], []
for item in EVAL_ITEMS:
    q = item['question']
    docs = retrieve(q, k=TOPK)
    labels = [is_relevant(q, d.page_content) for d in docs]
    recalls.append(recall_at_k(labels))
    mrrs.append(mrr_fn(labels))
    ndcgs.append(ndcg_fn(labels))
    print(f'Q: {q[:30]}... R={recalls[-1]:.0f} MRR={mrrs[-1]:.2f} nDCG={ndcgs[-1]:.2f}')

print(f'\n平均: R={sum(recalls)/len(recalls):.2f} MRR={sum(mrrs)/len(mrrs):.2f} nDCG={sum(ndcgs)/len(ndcgs):.2f}')


Q: 道通科技各产品线的毛利率分别是多少？... R=0 MRR=0.00 nDCG=0.00
Q: 公司前五大客户占比多少？... R=1 MRR=0.20 nDCG=0.20
Q: 研发费用和营收增长的关系是什么？... R=1 MRR=0.50 nDCG=0.44

平均: R=0.67 MRR=0.23 nDCG=0.21


## RAGAS


In [14]:
# RAGAS
try:
    from ragas import evaluate
    from ragas.llms import LangchainLLMWrapper
    from ragas.embeddings import LangchainEmbeddingsWrapper
    from datasets import Dataset

    # ragas==0.4.x 顶层导出的对象是可直接传给 evaluate 的 metric 实例
    from ragas.metrics import answer_relevancy, faithfulness, context_precision

    ragas_llm = LangchainLLMWrapper(llm)
    ragas_emb = LangchainEmbeddingsWrapper(emb)

    ANSWER_PROMPT = ChatPromptTemplate.from_messages([
        ('system', '基于上下文回答。'),
        ('human', '问题：{q}\n\n上下文：{ctx}')
    ])

    def rag_answer(q, k=6):
        docs = retrieve(q, k=k)
        ctx = '\n\n'.join([f'[{i}] {d.page_content}' for i, d in enumerate(docs, 1)])
        ans = (ANSWER_PROMPT | llm).invoke({'q': q, 'ctx': ctx}).content.strip()
        contexts = [d.page_content for d in docs]
        return ans, contexts

    print('构建数据...')
    records = []
    for item in EVAL_ITEMS:
        q = item['question']
        ref = item['reference']
        ans, ctxs = rag_answer(q, k=TOPK)
        records.append({
            'user_input': q,
            'response': ans,
            'retrieved_contexts': ctxs,
            'reference': ref,
        })
        print(f'  {q[:30]}... OK')

    ds = Dataset.from_list(records)
    print('RAGAS评估...')

    result = evaluate(
        ds,
        llm=ragas_llm,
        embeddings=ragas_emb,
        metrics=[answer_relevancy, faithfulness, context_precision]
    )
    print(result)

except Exception as e:
    print(f'错误: {e}')
    import traceback
    traceback.print_exc()


构建数据...
  道通科技各产品线的毛利率分别是多少？... OK
  公司前五大客户占比多少？... OK
  研发费用和营收增长的关系是什么？... OK
RAGAS评估...


Evaluating:   0%|          | 0/9 [00:00<?, ?it/s]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating: 100%|██████████| 9/9 [01:32<00:00, 10.33s/it]


{'answer_relevancy': 0.2736, 'faithfulness': 0.6827, 'context_precision': 0.0667}
